In [ ]:
#Software installation with maybe lsl streaming 


#software install
'''1. Install Unicorn Suite from g.tec
   → comes with Unicorn Recorder + Bluetooth drivers
   → requires Windows 10 Pro 64-bit, English

2. Pair the Unicorn via Bluetooth
   Settings → Bluetooth → Add device → "UN-XXXX.XX.XX"
   The serial number is printed on the headset

3. Install Python dependencies
'''

#lsl

'''bash# Windows acquisition machine
pip install pylsl
# → install Unicorn LSL App separately from g.tec website

# Game/analysis machine (any OS)
pip install pylsl mne numpy scipy pygame scikit-learn'''

In [ ]:
#measurement (prottoll)
'''[0:00 – 2:00]  Baseline       eyes open, fixation cross, no task
[2:00 – 4:00]  Focus task     serial subtraction (e.g. 300 - 7 - 7 - 7...)
[4:00 – 5:00]  Rest
[5:00 – 7:00]  High load      dual task: subtraction + tracking cursor
[7:00 – 8:00]  Rest
[8:00 – 10:00] Repeat focus
'''

In [ ]:
import mne

# ── Load raw .bdf ──────────────────────────────────────────────
raw = mne.io.read_raw_bdf('my_recording.bdf', preload=True)

print(raw.info)           # channel names, samplerate, duration
print(raw.ch_names)       # ['FZ','C3','CZ','C4','PZ','PO7','OZ','PO8', ...]
print(raw.info['sfreq'])  # 250.0

# plot raw signal — opens interactive window
raw.plot(duration=10, n_channels=8, scalings='auto')

In [ ]:
# 1. pick only EEG channels (drops accelerometer, battery, counter)
raw.pick_types(eeg=True)

# 2. set standard electrode positions (needed for topomaps later)
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage, match_case=False)

# 3. re-reference to average (standard for EEG)
raw.set_eeg_reference('average', projection=True)
raw.apply_proj()

# 4. filter: 1–40 Hz bandpass + 50 Hz notch
raw.filter(l_freq=1.0, h_freq=40.0)
raw.notch_filter(freqs=50)

# 5. save as .fif for fast reloading later
raw.save('my_recording_preprocessed_raw.fif', overwrite=True)

In [ ]:
raw = mne.io.read_raw_fif('my_recording_preprocessed_raw.fif', preload=True)

In [ ]:
# Option A: manual epochs by time
# e.g. baseline = 0–120s, focus = 120–240s
from mne import create_info
import numpy as np

tmin, tmax = 0.0, 2.0   # 2-second epochs
sfreq = raw.info['sfreq']

# extract focus segment manually
focus_raw = raw.copy().crop(tmin=120, tmax=240)
baseline_raw = raw.copy().crop(tmin=0, tmax=120)

def make_epochs(segment, label, epoch_len=2.0):
    """Cut a raw segment into fixed-length epochs."""
    data, times = segment[:]   # (n_channels, n_samples)
    n_samples = int(epoch_len * sfreq)
    n_epochs  = data.shape[1] // n_samples
    epochs = np.array([
        data[:, i*n_samples:(i+1)*n_samples]
        for i in range(n_epochs)
    ])  # shape: (n_epochs, n_channels, n_samples)
    labels = np.full(n_epochs, label, dtype=int)
    return epochs, labels

X_focus,    y_focus    = make_epochs(focus_raw,    label=1)
X_baseline, y_baseline = make_epochs(baseline_raw, label=0)

X = np.concatenate([X_baseline, X_focus], axis=0)
y = np.concatenate([y_baseline, y_focus], axis=0)
# X shape: (n_trials, 8, 500)  ← ready for RiemannMDM

In [ ]:
#Quick sanity checks before ML

raw.compute_psd(fmin=1, fmax=40).plot(picks='eeg')
# you should see alpha peak around 8–12 Hz at PO7/OZ/PO8
# and relatively flat beta at C3/CZ/C4

# average alpha power across scalp — sanity check for electrode contact
raw.compute_psd(fmin=8, fmax=12).plot_topomap()
# should be strongest at occipital channels (PO7, OZ, PO8)
# if it's random noise everywhere → electrode contact problem

# find epochs with peak-to-peak amplitude > 100 µV (blinks, EMG)
reject = dict(eeg=100e-6)  # 100 µV in volts
epochs_clean = epochs.drop_bad(reject=reject)
print(f"Kept {len(epochs_clean)}/{len(epochs)} epochs after artifact rejection")

In [ ]:
from sklearn.model_selection import cross_val_score
from models import RiemannMDM

mdm = RiemannMDM(use_pca=False)

# quick cross-validation to check if signal is classifiable
scores = cross_val_score(mdm, X, y, cv=5, scoring='accuracy')
print(f"Accuracy: {scores.mean():.2f} ± {scores.std():.2f}")
# chance level for 2 classes = 0.50
# good EEG classification = 0.70–0.85